In [1]:
import pandas as pd
from pathlib import Path

In [ ]:
lmp_dir = Path("../data/raw/lmp")
files = sorted(lmp_dir.glob("*_da_expost_lmp.csv"))
print(len(files), "files found")

3653 files found


In [7]:
def load_flambeau_lmp(path):
    df = pd.read_csv(path, skiprows=4)
    row = df[(df["Node"] == "DPC.FLAMBEAU") & (df["Value"] == "LMP")]
    if row.empty:
        return None

    file_date = pd.to_datetime(path.name[:8], format="%Y%m%d")

    he_cols = [c for c in df.columns if c.startswith("HE ")]
    long = row.melt(id_vars=["Node", "Type", "Value"], value_vars=he_cols, var_name="HE", value_name="LMP")
    long["hour"] = long["HE"].str.replace("HE ", "").astype(int)
    long["timestamp"] = file_date + pd.to_timedelta(long["hour"] - 1, unit="h")
    return long[["timestamp", "LMP"]]

In [8]:
test = load_flambeau_lmp(files[0])
print(test)

             timestamp    LMP
0  2016-01-01 00:00:00  17.35
1  2016-01-01 01:00:00  17.50
2  2016-01-01 02:00:00  15.01
3  2016-01-01 03:00:00  13.93
4  2016-01-01 04:00:00  13.28
5  2016-01-01 05:00:00  15.42
6  2016-01-01 06:00:00  18.13
7  2016-01-01 07:00:00  19.20
8  2016-01-01 08:00:00  19.91
9  2016-01-01 09:00:00  19.91
10 2016-01-01 10:00:00  20.86
11 2016-01-01 11:00:00  20.76
12 2016-01-01 12:00:00  20.47
13 2016-01-01 13:00:00  19.74
14 2016-01-01 14:00:00  19.03
15 2016-01-01 15:00:00  19.07
16 2016-01-01 16:00:00  19.98
17 2016-01-01 17:00:00  23.58
18 2016-01-01 18:00:00  25.35
19 2016-01-01 19:00:00  24.06
20 2016-01-01 20:00:00  23.59
21 2016-01-01 21:00:00  22.15
22 2016-01-01 22:00:00  21.38
23 2016-01-01 23:00:00  20.82


In [11]:
cache_path = Path("../data/processed/lmp_hourly.csv")

if cache_path.exists():
    lmp_df = pd.read_csv(cache_path, parse_dates=["timestamp"])
else:
    all_lmp = []
    for f in files:
        result = load_flambeau_lmp(f)
        if result is not None:
            all_lmp.append(result)
        else:
            print("missing Flambeau row:", f.name)
    lmp_df = pd.concat(all_lmp, ignore_index=True).sort_values("timestamp").reset_index(drop=True)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    lmp_df.to_csv(cache_path, index=False)

print(lmp_df.shape)
lmp_df.head()

(87672, 2)


,timestamp,LMP
0,2016-01-01 00:00:00,17.35
1,2016-01-01 01:00:00,17.50
2,2016-01-01 02:00:00,15.01
3,2016-01-01 03:00:00,13.93
4,2016-01-01 04:00:00,13.28
